In [15]:
## TO RUN ON THE CLOUD 

## preprocessed the data before starting the slide
## import embeddings <s
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [16]:
import fiftyone as fo

## list all dataset
print(fo.list_datasets())

# Close any zombie sessions that might be hanging
fo.close_app()

['dugong']


In [17]:

from pathlib import Path
import os
import numpy as np
import pandas as pd
import json
from fiftyone import ViewField as F
import matplotlib.pyplot as plt
import numpy as np

import fiftyone.brain as fob
from sklearn.preprocessing import normalize
import plotly.express as px
import skdim
import seaborn as sns
import pandas as pd
from sklearn.manifold import TSNE
import cv2
# ## renders plotly properly in a html instance. 
# import plotly.io as pio
# pio.renderers.default = "notebook"

In [18]:
# load dataset
dataset = fo.load_dataset("dugong")

In [19]:
## load views
nc_view = dataset.load_saved_view('nc')
wp_view = dataset.load_saved_view('wp')

## load saved test view
test_view_seed0 = dataset.load_saved_view('wp_test_s0')
test_view_seed63 = dataset.load_saved_view('wp_test_s63')
test_view_seed72 =  dataset.load_saved_view('wp_test_s72')

In [20]:
dataset.distinct('tags')

['notin_TEST_0',
 'notin_TEST_63',
 'notin_TEST_72',
 'notin_val_0',
 'notin_val_63',
 'notin_val_72',
 'test_0',
 'test_63',
 'test_72',
 'train_0',
 'train_63',
 'train_72',
 'val_0',
 'val_63',
 'val_72']

In [21]:
val_seed0 = wp_view.match_tags('val_0')
val_seed63 = wp_view.match_tags('val_63')
val_seed72 = wp_view.match_tags('val_72')

## get ebeddings 
val_seed0_emb = val_seed0.values("full_embeddings")
val_seed63_emb = val_seed63.values("full_embeddings")
val_seed72_emb = val_seed72.values("full_embeddings")

## convert to array and normalize
val_seed0_emb = normalize(np.array(val_seed0_emb))
val_seed63_emb = normalize(np.array(val_seed63_emb))
val_seed72_emb = normalize(np.array(val_seed72_emb))


# LID at the test set

In [25]:
## get embeddings for each view and normalize it 
## retrieve values
#nc_emb = nc_view.values("full_embeddings")

test_s0_emb = test_view_seed0.values("full_embeddings")
test_s63_emb = test_view_seed63.values("full_embeddings")
test_s72_emb = test_view_seed72.values("full_embeddings")

## convert to array
test_s0_emb = np.array(test_s0_emb)
test_s63_emb = np.array(test_s63_emb)
test_s72_emb = np.array(test_s72_emb)

## normalize 
test_s0_emb = normalize(test_s0_emb)
test_s63_emb = normalize(test_s63_emb)
test_s72_emb = normalize(test_s72_emb)


In [9]:
import time
from contextlib import contextmanager

@contextmanager
def timer(name):
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"{name}: {end - start:.4f} s")
    
def calculate_LID(dataset: np.ndarray, 
                  list_neighboors: list = [3,5,10,20],
                  verbose=True):
    """
    Computes Maximum Likelihood Estimation for a set of neighboors,
    Returns:
        list of MLE distances for each neighboor
    """
    def compute(dataset, 
                n_neighboors: int,
                n_jobs: int = 1):
        """
        Returns a float containing the intrinsic measure. 
        """
        mlea = skdim.id.MLE().fit_transform(X = dataset,
                                                n_neighbors = n_neighboors,
                                                n_jobs = 1)
        return mlea 
        
    results = {}
    for k in list_neighboors:
        if verbose:
            print(f"running {k}")
            with timer(f'k_{k}'):
                mlea = compute(dataset,
                               n_neighboors = k)
                results[k] = mlea
        else:
            mlea = compute(dataset,
                           n_neighboors = k)
            results[k] = mlea
    return results


In [24]:
out_results = {}
for i,emb in zip(('seed0','seed63','seed72'),(test_s0_emb, test_s63_emb, test_s72_emb)):
    print(f"calculating seed: {i}")
    results = calculate_LID(
        dataset = emb,
        list_neighboors = [25]
    )
    out_results[i] = results

calculating seed: seed0
running 25


k_25: 0.5590 s
calculating seed: seed63
running 25
k_25: 0.5154 s
calculating seed: seed72
running 25
k_25: 0.4176 s


In [25]:
out_results

{'seed0': {25: np.float64(6.116364506751639)},
 'seed63': {25: np.float64(4.390763114123529)},
 'seed72': {25: np.float64(5.853648455884514)}}

# LID at the validation set

In [13]:
out_results = {}
for i,emb in zip(('seed0','seed63','seed72'),(val_seed0_emb, val_seed63_emb, val_seed72_emb)):
    print(f"calculating seed: {i}")
    results = calculate_LID(
        dataset = emb,
        list_neighboors = [20,25,30]
    )
    out_results[i] = results

calculating seed: seed0
running 20


k_20: 0.4798 s
running 25
k_25: 0.5380 s
running 30
k_30: 0.5426 s
calculating seed: seed63
running 20
k_20: 0.5032 s
running 25
k_25: 0.6073 s
running 30
k_30: 0.4255 s
calculating seed: seed72
running 20
k_20: 0.4147 s
running 25
k_25: 0.4707 s
running 30
k_30: 0.6754 s


In [14]:
out_results

{'seed0': {20: np.float64(4.515314704889757),
  25: np.float64(4.1316495569352645),
  30: np.float64(3.85083927308934)},
 'seed63': {20: np.float64(4.2139626582388106),
  25: np.float64(4.001311788923311),
  30: np.float64(3.9025516480778903)},
 'seed72': {20: np.float64(4.711904081575396),
  25: np.float64(4.173857467621769),
  30: np.float64(3.953665744389702)}}

# Optimal Transport 

Wassertein Distance

In [22]:
import ot

In [23]:
nc_emb = nc_view.values('full_embeddings')
nc_emb_norm = normalize(np.array(nc_emb))

In [30]:
nc_emb = nc_view.take(100).values('full_embeddings')
nc_emb_norm = normalize(np.array(nc_emb))

In [31]:
## Cost Matrix of Euclidean Distance
source = nc_emb_norm

for i in zip(('test_0','test_63','test_72'),(test_s0_emb, test_s63_emb, test_s72_emb)):
    print(f"running:{i[0]}")
    target = i[1]
    M = ot.dist(source,
                target,
                metric='cosine')

    M /= M.max()  # normalize

    ## The Wassertein distance of NC and WP
    # Uniform weights — each point gets equal mass, summing to 1
    a = ot.unif(source.shape[0])   
    b = ot.unif(target.shape[0]) 

    # Wasserstein distance
    W = ot.emd2(a, b, M)
    print(f"Wassertein Distance: {W:4f}")

running:test_0
Wassertein Distance: 0.489119
running:test_63
Wassertein Distance: 0.389137
running:test_72
Wassertein Distance: 0.513352


In [32]:
## Cost Matrix of Euclidean Distance
source = nc_emb_norm

for i in zip(('test_0','test_63','test_72'),(test_s0_emb, test_s63_emb, test_s72_emb)):
    print(f"running:{i[0]}")
    target = i[1]
    M = ot.dist(source,
                target,
                metric='sqeuclidean')

    M /= M.max()  # normalize

    ## The Wassertein distance of NC and WP
    # Uniform weights — each point gets equal mass, summing to 1
    a = ot.unif(source.shape[0])   
    b = ot.unif(target.shape[0]) 

    # Wasserstein distance
    W = ot.emd2(a, b, M)
    print(f"Wassertein Distance: {W:4f}")

running:test_0
Wassertein Distance: 0.489119
running:test_63
Wassertein Distance: 0.389137
running:test_72
Wassertein Distance: 0.513352
